# LangChain Tool Calling & Agents — Beginner-Friendly Hands-On

This notebook builds an agent **step by step**, instead of starting with a large complicated application.

## Topics covered

1. Tool Calling
2. Agents
3. Error Handling
4. Memory with Checkpointer
5. PII Middleware
6. Tool Retry
7. Model Retry
8. Summarization Middleware
9. Final combined agent

### Learning path

```text
Simple LLM
   ↓
Tool Calling
   ↓
Agent
   ↓
Error Handling
   ↓
Memory + Checkpointer
   ↓
PII Protection
   ↓
Tool Retry + Model Retry
   ↓
Summarization
   ↓
More Reliable Agent
```

Every important code section includes comments explaining **what** we are doing and **why**.

## 0. Install packages

Run this once in a fresh environment.

> If the packages are already installed, you can skip this cell.

In [1]:
# Main packages:
# langchain              -> agent framework
# langgraph              -> agent runtime/checkpointing
# langchain-google-genai -> Gemini integration

# %pip install -U langchain langgraph langchain-google-genai

# 1. Imports and Gemini model

We use Gemini/OpenRouter as the LLM.

**Security note:** never hard-code a real API key into a notebook you share.

In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
# Create the Gemini/Gpt chat model.

# temperature=0 makes the output more deterministic

from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openai/gpt-oss-20b",
   
    temperature=0
)
print("OpenRouter model created successfully.")

# Part 2 — Tool Calling

## What is a tool?

A **tool is a Python function that the model/agent is allowed to call**.

Example:

```text
User: What is 12 × 5?
          ↓
        Model
          ↓
   "I should use multiply"
          ↓
   multiply(12, 5)
          ↓
          60
          ↓
     Final answer
```

The model does not automatically execute arbitrary Python. We explicitly give it tools.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

# First test the normal Python function.
print(multiply.invoke({"a": 6, "b": 7}))

42


In [ ]:
# bind_tools() tells the model that it is allowed to request this tool.
model_with_tools = model.bind_tools([multiply])

response = model_with_tools.invoke(
    "What is 12 multiplied by 5?"
)

print(response)

content='' additional_kwargs={} response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1794569079-1KOcvZpJhZyBMB0QlLKb', 'created': 1794569079, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 3.734e-06, 'cost_details': {'upstream_inference_completions_cost': 1.056e-06, 'upstream_inference_prompt_cost': 2.678e-06, 'upstream_inference_cost': 3.734e-06}, 'system_fingerprint': 'vllm-0.32.0-8589aa2a'} id='lc_run--01a1d4a7-0861-70c3-840a-99af3700fd89-0' tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 5}, 'id': 'chatcmpl-tool-ab3f441889d50a2e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_tokens': 32, 'total_tokens': 161, 'input_token_details': {'cache_read': 80, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


### Tool calling vs executing the tool

`bind_tools()` gives the model the **ability to request** a tool.

A complete application still needs to execute the requested tool and send its result back to the model.

That repeated process is one of the main things an **agent** manages for us.

# Part 3 — Agents

## What is an agent?

A simple mental model:

```text
User question
      ↓
    Agent
      ↓
  ┌───┴────────────┐
  ↓                ↓
Answer directly   Call a tool
                     ↓
                 Tool result
                     ↓
                   Agent
                     ↓
               Final answer
```

The agent can decide which tool to use and can make multiple tool calls when needed.

In [ ]:
from langchain.agents import create_agent

# create_agent() gives us the agent loop.
math_agent = create_agent(
    model=model,
    tools=[multiply],
)

print("Agent created.")

Agent created.


In [ ]:
result = math_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 15 multiplied by 8?"}
    ]
})

print(result["messages"][-1].content)

15 multiplied by 8 equals **120**.


In [ ]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

# Now the agent has two capabilities.
math_agent = create_agent(
    model=model,
    tools=[add, multiply],
)

result = math_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add 25 and 17, then multiply the result by 2."
        }
    ]
})

print(result["messages"][-1].content)

The result is **84**.


### Conceptual ReAct loop

Agents are often explained using the ReAct idea:

```text
Decide → Act → Observe → Decide → Act → ... → Final answer
```

For the previous example:

```text
add(25, 17)
     ↓
42
     ↓
multiply(42, 2)
     ↓
84
```

We are describing the decision loop conceptually; we are **not exposing private chain-of-thought**.

# Part 4 — Error Handling

## Why do agents need error handling?

Tools can fail:

- an API can be unavailable
- a database can reject a request
- input can be invalid
- a Python function can raise an exception

Without handling, one tool exception can make the whole request fail.

LangChain provides `ToolErrorMiddleware` to convert tool failures into controlled tool messages that the agent can respond to.

In [ ]:
from langchain.agents.middleware import ToolErrorMiddleware

def explain_tool_error(exc, request):
    # Convert the Python exception into useful information for the agent.
    return (
        f"The tool `{request.tool_call['name']}` failed with "
        f"{type(exc).__name__}. Check the inputs and try again."
    )

error_middleware = ToolErrorMiddleware(
    on_error=explain_tool_error
)

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. The divisor must not be zero."""

    if b == 0:
        # This represents a real tool failure.
        raise ValueError("Cannot divide by zero.")

    return a / b


error_agent = create_agent(
    model=model,
    tools=[divide],
    middleware=[error_middleware],
)

print("Error-handling agent created.")

Error-handling agent created.


In [ ]:
# The tool can fail, but middleware gives the agent a controlled error message.
try:
    result = error_agent.invoke({
        "messages": [
            {"role": "user", "content": "Calculate 10 divided by 0."}
        ]
    })

    print(result["messages"][-1].content)

except Exception as e:
    print("Request ended with an exception:", e)

Dividing by zero is undefined in mathematics. In most programming languages and calculators, attempting to compute \(10 \div 0\) will raise an error or return an infinite/undefined result. If you need to handle such cases, you should check whether the divisor is zero before performing the division and decide how to respond (e.g., return an error message, `null`, or a special value).


In [ ]:
def divide(a: float, b: float) -> float:
    """Divide a by b. The divisor must not be zero."""

    if b == 0:
        # This represents a real tool failure.
        raise ValueError("Cannot divide by zero.")

    return a / b

divide(10,0)

ValueError: Cannot divide by zero.

### Important distinction

**Error handling is not the same as retry.**

- Error handling → "What should happen after a tool fails?"
- Retry → "Should we try the operation again because the failure may be temporary?"

# Part 5 — Memory with Checkpointer

## Why memory?

Without memory:

```text
User: My name is Rahul.
Agent: Nice to meet you.

User: What is my name?
Agent: I don't know.
```

With short-term thread memory:

```text
User: My name is Rahul.
Agent: Nice to meet you.

User: What is my name?
Agent: Your name is Rahul.
```

A **checkpointer** saves the agent's state for a conversation thread.

Think of:

```text
thread_id = conversation ID
```

In [ ]:
# python -m pip install --upgrade langgraph langgraph-checkpoint

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver is excellent for teaching and demos.

# For production, use a persistent checkpointer/database. - Postgresql
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=model,
    tools=[add, multiply],
    checkpointer=checkpointer,
)

config = {
    "configurable": {
        "thread_id": "student-1"
    }
}

In [ ]:
# First message in this conversation.
memory_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Hi! My name is praveen."}
        ]
    },
    config,
)

print("Message stored in student-1 thread.")

Message stored in student-1 thread.


In [ ]:
# Same thread_id = same conversation state.
result = memory_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my name?"}
        ]
    },
    config,
)

print(result["messages"][-1].content)

Your name is Praveen.


In [ ]:
# A different thread starts a different conversation.
new_config = {
    "configurable": {
        "thread_id": "student-2"
    }
}

result = memory_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my name?"}
        ]
    },
    new_config,
)

print(result["messages"][-1].content)

I’m not sure—could you tell me your name?


In [ ]:
# First message in this conversation.
memory_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Hi! My name is Aman."}
        ]
    },
    new_config,
)

print("Message stored in student-2 thread.")

Message stored in student-2 thread.


In [ ]:
# Same thread_id = same conversation state.
result = memory_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my name?"}
        ]
    },
    new_config,
)

print(result["messages"][-1].content)

Your name is Aman.


### Memory summary

```text
Agent
  +
Checkpointer
  +
thread_id
  ↓
Conversation-specific short-term memory
```

`InMemorySaver` is not durable storage. It is mainly useful for learning and development.

# Part 6 — PII Middleware

## What is PII?

PII means **Personally Identifiable Information**.

Examples:

- email addresses
- credit-card numbers
- IP addresses
- URLs

Sometimes an application should protect sensitive information before it reaches the model or tools.

LangChain provides `PIIMiddleware`.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

# Redact email addresses from user input.
email_redaction = PIIMiddleware(
    "email",
    strategy="redact",
    apply_to_input=True,
)

pii_agent = create_agent(
    model=model,
    tools=[],
    middleware=[email_redaction],
)

print("PII-protected agent created.")

PII-protected agent created.


In [ ]:
# Use fake data for the classroom demo.
result = pii_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "My email is student@example.com. "
                "Please say hello to me."
            )
        }
    ]
})

print(result["messages"][-1].content)

Hello! 👋


## PII strategies

| Strategy | Meaning |
|---|---|
| `redact` | Replace sensitive value with a placeholder |
| `mask` | Partially hide the value |
| `hash` | Replace it with a deterministic hash |
| `block` | Reject the request |

Other supported built-in PII detectors include email, credit card, IP, MAC address, and URL.



# Part 7 — Tool Retry

## Why retry?

Some failures are temporary:

```text
Agent → External API
          ↓
       Timeout
          ↓
      Try again
          ↓
        Success
```

Retry can help with transient network/API problems.

Do **not** blindly retry every error. Invalid input normally needs correction, not another identical attempt.

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

tool_retry = ToolRetryMiddleware(
    max_retries=3,
    backoff_factor=2.0,
    initial_delay=1.0,
)

retry_agent = create_agent(
    model=model,
    tools=[divide],
    middleware=[tool_retry],
)

print("Tool-retry agent created.")

Tool-retry agent created.


## A flaky tool for demonstration

We will deliberately fail twice and succeed on the third attempt.

This makes the purpose of retry easy to see in class.

In [ ]:
attempts = {"count": 0}

@tool
def flaky_lookup() -> str:
    """Demo tool that temporarily fails before succeeding."""

    attempts["count"] += 1

    if attempts["count"] < 3:
        raise TimeoutError("Temporary service timeout.")

    return "Service is available now."

In [ ]:
flaky_retry_agent = create_agent(
    model=model,
    tools=[flaky_lookup],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=1.0,
            initial_delay=0.1,
        )
    ],
)

attempts["count"] = 0

try:
    result = flaky_retry_agent.invoke({
        "messages": [
            {"role": "user", "content": "Check the service status."}
        ]
    })

    print(result["messages"][-1].content)
    print("Tool attempts:", attempts["count"])

except Exception as e:
    print("Request failed:", e)

✅ **Service Status:** All systems are up and running. Let me know if you need anything else!
Tool attempts: 3


### Expected conceptual flow

```text
Attempt 1 → Timeout
Attempt 2 → Timeout
Attempt 3 → Success
```

So:

```text
Error handling → deal with failure
Retry          → attempt the operation again
```

# Part 8 — Model Retry

Tools are not the only things that can fail.

The model/API request can also fail because of temporary network problems, service errors, or rate limits.

`ModelRetryMiddleware` lets us configure retry behavior around model calls.

In [ ]:
from langchain.agents.middleware import ModelRetryMiddleware

model_retry = ModelRetryMiddleware(
    max_retries=3,
    backoff_factor=2.0,
    initial_delay=1.0,
)

model_retry_agent = create_agent(
    model=model,
    tools=[add, multiply],
    middleware=[model_retry],
)

print("Model-retry agent created.")

Model-retry agent created.


### Why don't we intentionally break Gemini?

A real API outage is unpredictable and is not a good classroom test.

Instead, you should understand the configuration:

```text
Model call
   ↓
Temporary failure?
   ↓ yes
Retry
   ↓
Success → continue
   ↓
Still failing → return failure
```

Retry should be used for errors that are actually retryable.

# Part 9 — Combining Tool Retry + Error Handling

A useful production-style pattern is:

```text
Tool call
   ↓
Retry temporary failure
   ↓
Still failing?
   ↓
Convert final failure into a controlled error
```

This gives transient failures a chance to recover while still preventing an uncontrolled tool exception from crashing the agent.

In [ ]:
combined_agent = create_agent(
    model=model,
    tools=[divide],
    middleware=[
        # Retry first. If retries are exhausted, pass the error onward.
        ToolRetryMiddleware(
            max_retries=2,
            backoff_factor=1.0,
            initial_delay=0.2,
            on_failure="error",
        ),

        # Handle the final tool error.
        ToolErrorMiddleware(
            on_error=explain_tool_error
        ),
    ],
)

print("Combined retry + error-handling agent created.")

Combined retry + error-handling agent created.


# Part 10 — Summarization Middleware

## Why summarize conversation history?

A long conversation can become:

```text
Message 1
Message 2
Message 3
...
Message 100
```

Sending all old messages every time can increase context size, latency, and cost.

Summarization middleware can compress older conversation history while keeping recent messages.

```text
Long conversation
       ↓
Summarization
       ↓
Older messages → summary
Recent messages → kept
       ↓
Smaller useful context
```

In [ ]:
# Import the OpenRouter chat model.
from langchain_openrouter import ChatOpenRouter

# Import middleware for automatically summarizing long conversations.
from langchain.agents.middleware import SummarizationMiddleware

# Create a separate model for summarizing conversations.
summary_model = ChatOpenRouter(
    model="openai/gpt-oss-20b",  # Model used for summarization.
    temperature=0                # Keep summaries consistent.
)

# Create the agent with summarization and memory.
summarization_agent = create_agent(
    model=model,                 # Main model used by the agent.
    tools=[add, multiply],       # Tools available to the agent.

    # Add summarization middleware to manage long conversations.
    middleware=[
        SummarizationMiddleware(
            model=summary_model,       # Model used for summarization.
            trigger=("tokens", 4000),  # Summarize when tokens reach 4000.
            keep=("messages", 20),     # Keep the latest 20 messages.
        )
    ],

    # Save conversation state in memory.
    checkpointer=InMemorySaver(),
)

# Confirm that the agent was created successfully.
print("Summarization-enabled agent created.")

Summarization-enabled agent created.


### Memory vs summarization

They solve different problems:

- **Checkpointer** → stores conversation state for a thread.
- **Summarization** → compresses older history when the conversation becomes large.

They can be used together.

# Part 11 — Final Combined Agent

Now we combine everything into one beginner-friendly agent.

Features:

- Tools
- Agent loop
- Tool error handling
- Tool retry
- Model retry
- PII protection
- Checkpointer memory
- Summarization

In [ ]:
from langchain_openrouter import ChatOpenRouter

# Create the model used for conversation summarization.
summary_model = ChatOpenRouter(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Create a fresh checkpointer for the final demo.
final_checkpointer = InMemorySaver()

# Protect email addresses in user input.
final_pii = PIIMiddleware(
    "email",
    strategy="redact",
    apply_to_input=True,
)

# Retry temporary tool failures.
final_tool_retry = ToolRetryMiddleware(
    max_retries=2,
    backoff_factor=2.0,
    initial_delay=0.5,
    on_failure="error",
)

# Retry temporary model/API failures.
final_model_retry = ModelRetryMiddleware(
    max_retries=2,
    backoff_factor=2.0,
    initial_delay=1.0,
)

# Handle a tool error after retries are exhausted.
final_tool_error = ToolErrorMiddleware(
    on_error=explain_tool_error
)

# Summarize older messages when the conversation becomes large.
final_summarization = SummarizationMiddleware(
    model=summary_model,
    trigger=("tokens", 4000),
    keep=("messages", 20),
)

# Create the final agent with all middleware features.
final_agent = create_agent(
    model=model,
    tools=[add, multiply, divide],
    middleware=[
        final_pii,
        final_model_retry,
        final_tool_retry,
        final_tool_error,
        final_summarization,
    ],
    checkpointer=final_checkpointer,
)

# Confirm that the final combined agent was created successfully.
print("Final combined agent created.")

Final combined agent created.


In [ ]:
# thread_id identifies this conversation.
final_config = {
    "configurable": {
        "thread_id": "demo-student-1"
    }
}

result = final_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Hi! My name is Ajay. "
                    "My email is student@example.com. "
                    "What is 12 multiplied by 8?"
                ),
            }
        ]
    },
    final_config,
)

print(result["messages"][-1].content)

Hi Ajay! 12 multiplied by 8 equals **96**.


In [ ]:
# Same thread_id → the agent can use the previous conversation state.
result = final_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my name?"}
        ]
    },
    final_config,
)

print(result["messages"][-1].content)

Your name is **Ajay**.


In [ ]:
# The agent can decide to use multiple tools.
result = final_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Add 45 and 55, then multiply the result by 3."
            }
        ]
    },
    final_config,
)

print(result["messages"][-1].content)

The calculation gives **300**.


# Part 12 — Architecture

```text
                         USER
                           │
                           ▼
                    ┌─────────────┐
                    │    AGENT    │
                    └──────┬──────┘
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
        Model/API call             Tool call
              │                         │
        Model Retry                Tool Retry
              │                         │
              ▼                         ▼
        Model response          Tool execution
                                        │
                                  Tool Error
                                  Handling
              │                         │
              └────────────┬────────────┘
                           ▼
                     Final response

        ┌────────────────────────────────────┐
        │ Middleware                         │
        │                                    │
        │ PII Protection                     │
        │ Model Retry                        │
        │ Tool Retry                         │
        │ Tool Error Handling                │
        │ Summarization                      │
        └────────────────────────────────────┘

        ┌────────────────────────────────────┐
        │ Checkpointer                       │
        │                                    │
        │ thread_id → conversation state     │
        └────────────────────────────────────┘
```

# Part 13 — Real-World Analogy

Imagine a customer-support AI:

### Tool Calling
It can call:

- order database
- refund system
- shipping API

### Agent
It decides which tool to use.

### Error Handling
If a tool fails, the agent gets a controlled error instead of crashing.

### Tool Retry
If a shipping API temporarily times out, try again.

### Model Retry
If the AI API temporarily fails, retry the model call.

### PII Middleware
Protect sensitive customer information.

### Checkpointer
Remember what the customer said earlier in the same conversation.

### Summarization
If the conversation becomes very long, compress old history while keeping recent context.

# Part 14 — Student Challenges

## Challenge 1 — Add subtraction

Create a `subtract(a, b)` tool and add it to the agent.

## Challenge 2 — Create a risky tool

Make a tool that raises `ValueError` for invalid input. Add `ToolErrorMiddleware`.

## Challenge 3 — Memory

Create two different `thread_id` values and show that they have separate conversations.

## Challenge 4 — PII

Try another supported PII detector. Use only fake test data.

## Challenge 5 — Retry

Create a tool that fails twice and succeeds on the third attempt.

## Challenge 6 — Final project

Build a **Student Assistant Agent** with:

- at least 2 tools
- memory
- PII protection
- tool retry
- model retry
- error handling
- summarization

# Part 15 — Troubleshooting

### API key error

Set `GOOGLE_API_KEY` before creating the Gemini model.

### Import error after installing packages

Restart the notebook kernel after a major package upgrade, then run the cells again.

### `InMemorySaver` does not survive a restart

That is expected. It is in-memory storage for demos.

### Retry seems slow

Backoff intentionally adds waiting time. Classroom demos can use smaller delays.

### Summarization does not happen immediately

It only activates when the configured trigger is reached.

### Version differences

LangChain evolves quickly. If a middleware import or argument differs in your installed version, check the current LangChain documentation and upgrade compatible packages.

# Final Recap

The goal is to understand **why** each feature exists, not memorize API names.

```text
Tool Calling
    ↓
Give the model capabilities

Agent
    ↓
Let the model decide when/how to use those capabilities

Error Handling
    ↓
Recover from tool failures

Checkpointer
    ↓
Remember conversation state

PII Middleware
    ↓
Protect sensitive information

Tool Retry
    ↓
Retry temporary tool failures

Model Retry
    ↓
Retry temporary model/API failures

Summarization
    ↓
Keep long conversations manageable
```

### One-line mental model

> **An agent is a decision-making loop around a model and tools; middleware adds reliability, safety, memory, and control around that loop.**